In [12]:
import numpy as np
import pandas as pd

In [13]:
np.random.seed(42)
N = 400
X = np.random.uniform(-2, 2, (N, 3))
y = (np.sin(X[:, 0]) + 0.5 * (X[:, 1] ** 2) - 0.8 * X[:, 2]).reshape(-1, 1)

In [14]:
def relu(Z):
    return np.maximum(0, Z)

def drelu(Z):
    return (Z > 0).astype(float)

def sigmoid(Z):
    Zc = np.clip(Z, -500, 500)
    return 1.0 / (1.0 + np.exp(-Zc))

def dsigmoid(Z):
    s = sigmoid(Z); return s * (1 - s)

ACTS = {'relu': (relu, drelu), 'sigmoid': (sigmoid, dsigmoid)}

In [15]:
class DenseNet:
    def __init__(self, sizes, act='relu', seed=42):
        np.random.seed(seed)
        self.sizes = sizes
        self.L = len(sizes) - 1
        self.act, self.dact = ACTS[act]
        self.params = {}
        for l in range(self.L):
            out, inn = sizes[l+1], sizes[l]
            limit = np.sqrt(6.0 / (inn + out))
            self.params[f'W{l}'] = np.random.uniform(-limit, limit, (out, inn))
            self.params[f'b{l}'] = np.zeros((out,))

    def forward(self, A0):
        A = A0.copy()
        cache = {'A0': A}
        for l in range(self.L):
            W = self.params[f'W{l}']; b = self.params[f'b{l}']
            Z = A @ W.T + b
            cache[f'Z{l}'] = Z
            if l == self.L - 1:
                A = Z
            else:
                A = self.act(Z)
            cache[f'A{l+1}'] = A
        return A, cache

    def backward(self, X, y, cache):
        N = X.shape[0]
        grads = {}
        A_final = cache[f'A{self.L}']
        dA = (2.0 / N) * (A_final - y)
        for l in reversed(range(self.L)):
            Z = cache[f'Z{l}']
            A_prev = cache[f'A{l}']
            if l == self.L - 1:
                dZ = dA
            else:
                dZ = dA * self.dact(Z)
            dW = dZ.T @ A_prev / N
            db = np.sum(dZ, axis=0) / N
            grads[f'dW{l}'] = dW
            grads[f'db{l}'] = db
            dA = dZ @ self.params[f'W{l}']
        return grads

    def step(self, grads, lr):
        for l in range(self.L):
            self.params[f'W{l}'] -= lr * grads[f'dW{l}']
            self.params[f'b{l}'] -= lr * grads[f'db{l}']


In [16]:
def param_count(sizes):
    total = 0
    for i in range(len(sizes)-1):
        total += sizes[i+1]*sizes[i] + sizes[i+1]
    return total

In [17]:
def train_once(sizes, act='relu', lr=0.01, epochs=1000, record_epoch=200):
    net = DenseNet(sizes, act=act, seed=42)
    losses = []
    loss_at_record = None
    for ep in range(1, epochs+1):
        y_hat, cache = net.forward(X)
        loss = np.mean((y_hat - y)**2)
        losses.append(loss)
        grads = net.backward(X, y, cache)
        net.step(grads, lr)
        if ep == record_epoch:
            loss_at_record = loss
    y_hat, cache = net.forward(X)
    grads = net.backward(X, y, cache)
    if net.L <= 1:
        first_hidden = last_hidden = 0
    else:
        first_hidden = 0
        last_hidden = net.L - 2
    def grad_norm(idx):
        dW = grads[f'dW{idx}']; db = grads[f'db{idx}']
        v = np.concatenate([dW.ravel(), db.ravel()])
        return np.sqrt(np.sum(v**2))
    gn_first = grad_norm(first_hidden)
    gn_last = grad_norm(last_hidden)
    behavior = 'stable' if losses[-1] < 0.5*losses[0] else ('slow' if losses[-1] >= 0.9*losses[0] else 'unstable')
    return {
        'Params': param_count(sizes),
        'Loss@200': float(loss_at_record),
        'FinalLoss': float(losses[-1]),
        'GradNormFirst': float(gn_first),
        'GradNormLast': float(gn_last),
        'Behavior': behavior
    }


In [18]:
models = {
    'A': [3, 4, 1],
    'B': [3, 6, 6, 1],
    'C': [3, 8, 8, 8, 8, 1],
    'D': [3] + [8]*8 + [1]
}

In [22]:
results = []
for name, sizes in models.items():
    for act in ['sigmoid', 'relu']:
        r = train_once(sizes, act=act, lr=0.01, epochs=1000, record_epoch=200)
        r.update({'Model': name, 'Activation': act})
        results.append(r)
        print(f"{name}, {act}, params={r['Params']}, loss@200={r['Loss@200']:.6f}, final={r['FinalLoss']:.6f}, grad1={r['GradNormFirst']:.3e}, gradlast={r['GradNormLast']:.3e}, {r['Behavior']}")

A, sigmoid, params=21, loss@200=3.966821, final=3.647170, grad1=2.408e-03, gradlast=2.408e-03, unstable
A, relu, params=21, loss@200=4.730731, final=4.086222, grad1=7.901e-03, gradlast=7.901e-03, unstable
B, sigmoid, params=73, loss@200=4.407568, final=3.849319, grad1=7.567e-04, gradlast=4.153e-03, unstable
B, relu, params=73, loss@200=3.861947, final=2.833448, grad1=6.035e-03, gradlast=1.197e-02, unstable
C, sigmoid, params=257, loss@200=2.101433, final=2.023650, grad1=3.379e-05, gradlast=1.249e-03, slow
C, relu, params=257, loss@200=2.737661, final=2.550004, grad1=2.014e-03, gradlast=3.688e-03, slow
D, sigmoid, params=545, loss@200=2.405976, final=2.242414, grad1=1.090e-07, gradlast=2.398e-03, slow
D, relu, params=545, loss@200=2.422460, final=2.261574, grad1=4.494e-05, gradlast=4.905e-03, slow


In [23]:
print("Summary table:")
df = pd.DataFrame(results)[['Model','Activation','Params','Loss@200','FinalLoss','GradNormFirst','GradNormLast','Behavior']]
print(df.to_string(index=False))

Summary table:
Model Activation  Params  Loss@200  FinalLoss  GradNormFirst  GradNormLast Behavior
    A    sigmoid      21  3.966821   3.647170   2.407504e-03      0.002408 unstable
    A       relu      21  4.730731   4.086222   7.900547e-03      0.007901 unstable
    B    sigmoid      73  4.407568   3.849319   7.567020e-04      0.004153 unstable
    B       relu      73  3.861947   2.833448   6.035268e-03      0.011970 unstable
    C    sigmoid     257  2.101433   2.023650   3.378685e-05      0.001249     slow
    C       relu     257  2.737661   2.550004   2.014283e-03      0.003688     slow
    D    sigmoid     545  2.405976   2.242414   1.089501e-07      0.002398     slow
    D       relu     545  2.422460   2.261574   4.494361e-05      0.004905     slow
